In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import os

def load_sql_dir(directory):
    records = []
    for file_name in sorted(os.listdir(directory)):
        if file_name.lower().endswith(".sql"):
            file_path = os.path.join(directory, file_name)
            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    sql_text = f.read()
                records.append({"file_name": file_name, "sql_text": sql_text})
            except (FileNotFoundError, PermissionError, UnicodeDecodeError) as e:
                print(f"[SKIP] {file_path} → {e}")
    
    df = pd.DataFrame(records, columns=["file_name", "sql_text"])
    print(f"Loaded {len(df)} .sql files from {directory}")
    return df

In [ ]:
import pandas as pd
import numpy as np
import sqlparse
import re
from gensim.models.doc2vec import Doc2Vec
from tqdm.auto import tqdm


class Doc2VecQueryEmbedder:
    def __init__(self, model_path, infer_epochs=20):
        """
        Parameters
        ----------
        model_path : str
            Path to trained Doc2Vec .model file
        infer_epochs : int
            Number of inference steps (higher = more accurate but slower)
        """
        print("Loading Doc2Vec model...")
        self.model = Doc2Vec.load(model_path)
        self.infer_epochs = infer_epochs

        self.sneaky_number_pattern = re.compile(r'^[-+]?(?:\d+\.\d*|\d*\.\d+|\d+)(?:[eE][-+]?\d+)?$')

    # ============================================================
    # SQL PREPROCESSING
    # ============================================================

    def _normalize_sql(self, sql):
        if not isinstance(sql, str):
            return ""
        sql = sql.lower().strip()
        return re.sub(r'\s+', ' ', sql)

    def _mask_literals(self, sql):
        sql = self._normalize_sql(sql)
        if not sql: return ""

        parsed = sqlparse.parse(sql)
        if not parsed: return ""

        tokens = []
        from sqlparse.tokens import Number, String

        for token in parsed[0].flatten():
            if token.is_whitespace: continue

            val = token.value

            if token.ttype in Number or \
               self.sneaky_number_pattern.fullmatch(val) or \
               re.fullmatch(r'^[-+]?\d+$', val):
                tokens.append('NUM_LITERAL')

            elif token.ttype in String: tokens.append('STR_LITERAL')

            elif val.startswith('0x') and len(val) > 2: tokens.append('HEX_LITERAL')

            else: tokens.append(val)

        return tokens

    # ============================================================
    # EMBEDDING GENERATION
    # ============================================================

    def generate_embeddings(self, df, text_column='sql_text'):
        """
        Generates Doc2Vec embeddings for SQL queries.

        Parameters
        ----------
        df : pd.DataFrame
        text_column : str

        Returns
        -------
        np.ndarray  shape = (num_queries, vector_size)
        """

        if text_column not in df.columns:
            raise ValueError(f"Column '{text_column}' not found in DataFrame.")

        print("Preprocessing SQL queries...")
        processed_queries = df[text_column].apply(self._mask_literals).tolist()

        print("Inferring Doc2Vec embeddings...")
        embeddings = []

        for tokens in tqdm(processed_queries, desc="Doc2Vec Inference"):
            if not tokens:
                embeddings.append(np.zeros(self.model.vector_size))
                continue

            vector = self.model.infer_vector(tokens,epochs=self.infer_epochs)
            embeddings.append(vector)

        return np.vstack(embeddings)

In [ ]:
# Load SQL dataframe
df_stackoverflow = load_sql_dir(path)

# Initialize embedder
doc2vec_embedder = Doc2VecQueryEmbedder(
    model_path="doc2vec_checkpoints/doc2vec_final.model",
    infer_epochs=20
)

# Generate embeddings
doc2vec_embeddings = doc2vec_embedder.generate_embeddings(df_stackoverflow,text_column="sql_text")

# Save
np.save("doc2vec_embeds.npy", doc2vec_embeddings)

print("Embeddings shape:", doc2vec_embeddings.shape)